# Electromagnetic Resonators and Electrodynamics: Equations and Derivation

This notebook uses a field-first textbook sequence:
**Maxwell equations -> cavity/material constraints -> modal reduction -> Hamiltonian structure -> data-facing model**.

Renderer note: Mermaid blocks depend on notebook renderer support. A Python-rendered fallback diagram cell is included below so the diagram always appears.

## Model Diagram (Electrodynamics to Resonator Data)
Mermaid source is kept for portability, but many notebook renderers do not embed Mermaid directly.
Run **Cell 2** to view the guaranteed rendered theory diagram.

## Full Maxwell Equations for the Model
1. $\nabla\cdot\mathbf{D}=\rho_f$.
2. $\nabla\cdot\mathbf{B}=0$.
3. $\nabla\times\mathbf{E}=-\partial\mathbf{B}/\partial t$.
4. $\nabla\times\mathbf{H}=\mathbf{J}_f+\partial\mathbf{D}/\partial t$.
5. With closure: $\mathbf{D}=\epsilon\mathbf{E}$, $\mathbf{B}=\mu\mathbf{H}$.

For source-free homogeneous media, these produce wave equations such as
$$
\nabla^2\mathbf{E}-\mu\epsilon\frac{\partial^2\mathbf{E}}{\partial t^2}=0.
$$

## Independent and Dependent Variables (and Why)
- Independent variables: space $(x,y,z)$ and time $t$ because fields propagate in spacetime.
- Dependent field variables: $\mathbf{E},\mathbf{H},\mathbf{D},\mathbf{B}$ because they encode energy storage and transport.
- Reduced dependent modal states: $a_n(t)$ after projection onto resonator modes.
- Why reduction: measured data are low-dimensional while physics is field-dimensional.

## Trivial Case Before Full Network
We solve the fundamental 1D cavity mode first:
1. $E(x,t)=E_0\sin(kx)\cos(\omega t)$.
2. $H(x,t)=\frac{E_0}{Z}\cos(kx)\sin(\omega t)$ (phase-shifted dual field).
3. Boundary condition gives $k=\pi/L$ for the first mode.
4. Dispersion: $\omega = v k$ with $v=1/\sqrt{\mu\epsilon}$.

The code cell below plots this trivial mode and an explicit model-flow diagram so visuals show up even without Mermaid rendering.

In [ ]:
# Theory diagram renderer (always visible in notebooks, independent of Mermaid support).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
blocks = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Model Evaluation / Data Fit'),
]
for x0, y0, label in blocks:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#eef4ff')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Theory Diagram (Rendered Fallback)')
plt.tight_layout()
plt.show()

## Variational and Hamiltonian Structure

### Field Lagrangian Density
A standard vacuum-like form is
$$
\mathcal{L}_{\text{field}} = \frac{\epsilon}{2}|\mathbf{E}|^2 - \frac{1}{2\mu}|\mathbf{B}|^2.
$$
Applying Hamilton's principle to this density with Maxwell constraints recovers wave dynamics.

### Single-Mode Reduction
1. Expand field in basis functions and keep dominant mode $a(t)$.
2. Reduced energy form:
   $$
   H(a,\pi_a)=\frac{\pi_a^2}{2M_{\text{eff}}}+\frac{K_{\text{eff}}}{2}a^2.
   $$
3. Canonical equations imply $\ddot a + \omega_0^2 a=0$ where $\omega_0=\sqrt{K_{\text{eff}}/M_{\text{eff}}}$.

### Data-Facing Variables in This Notebook
1. Independent fit variable: normalized index/time-like coordinate $x$.
2. Dependent observed variable: measured scalar signal $y(x)$.
3. Surrogate parameters $(A,\gamma,\omega,c_0)$ represent amplitude, damping, angular frequency, and baseline.
4. Interpretation: surrogate fit is validated against the mechanistic mode model, not used in isolation.

## Public Data Sources and Peer-Reviewed References

1. **NIST Microwave Resonator Data**
   - URL: https://www.nist.gov/
   - Peer-reviewed context: Use resonator datasets linked in peer-reviewed microwave metrology papers
2. **Stock Data 2 (CSV fallback)**
   - URL: https://raw.githubusercontent.com/plotly/datasets/master/stockdata2.csv
   - Peer-reviewed context: Public benchmark fallback

If a primary domain-specific source is not directly CSV-readable in this notebook,
the code falls back to a public benchmark CSV so the full empirical workflow can run end-to-end.


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import requests
import ipywidgets as widgets
from ipywidgets import interact

plt.style.use('seaborn-v0_8-whitegrid')

DOMAIN_KEY = '14_em_resonators'
DOMAIN_TITLE = 'EM Resonators'
MODEL_FAMILY = 'damped'
DATA_SOURCES = [
  {
    "name": "NIST Microwave Resonator Data",
    "url": "https://www.nist.gov/",
    "reference": "Use resonator datasets linked in peer-reviewed microwave metrology papers"
  },
  {
    "name": "Stock Data 2 (CSV fallback)",
    "url": "https://raw.githubusercontent.com/plotly/datasets/master/stockdata2.csv",
    "reference": "Public benchmark fallback"
  }
]


In [ ]:
# Fallback model diagram renderer (works even when Mermaid is not embedded by the notebook renderer).
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
nodes = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Observable Data Fit'),
]
for x0, y0, label in nodes:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#edf3fb')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Fallback Diagram (Python-rendered)')
plt.tight_layout()
plt.show()

In [ ]:
# Trivial electrodynamics mode: 1D cavity standing wave with an always-rendered diagram.
from matplotlib.patches import FancyBboxPatch

L_box = 1.0
eps0 = 1.0
mu0 = 1.0
v_wave = 1.0 / np.sqrt(mu0 * eps0)
k1 = np.pi / L_box
omega1 = v_wave * k1
E0 = 1.0
Z0 = np.sqrt(mu0 / eps0)

t_snap = 0.35
x_space = np.linspace(0.0, L_box, 500)
t_time = np.linspace(0.0, 4.0 * np.pi / omega1, 500)

E_x = E0 * np.sin(k1 * x_space) * np.cos(omega1 * t_snap)
H_x = (E0 / Z0) * np.cos(k1 * x_space) * np.sin(omega1 * t_snap)
E_t_mid = E0 * np.sin(k1 * (L_box / 2.0)) * np.cos(omega1 * t_time)
H_t_mid = (E0 / Z0) * np.cos(k1 * (L_box / 2.0)) * np.sin(omega1 * t_time)

print("Trivial cavity mode:")
print(f"k1 = pi/L = {k1:.5f}, omega1 = v*k1 = {omega1:.5f}, v = {v_wave:.3f}")
print("E(x,t)=E0 sin(kx) cos(omega t), H(x,t)=(E0/Z) cos(kx) sin(omega t)")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax0 = ax[0]
ax0.axis("off")
for x0, y0, txt in [
    (0.06, 0.74, "Maxwell PDEs"),
    (0.06, 0.48, "Boundary/Material"),
    (0.06, 0.22, "Mode ODE a(t)"),
    (0.56, 0.48, "Measured Response"),
]:
    patch = FancyBboxPatch((x0, y0), 0.34, 0.16, boxstyle="round,pad=0.02", linewidth=1.2, facecolor="#eef9f0")
    ax0.add_patch(patch)
    ax0.text(x0 + 0.17, y0 + 0.08, txt, ha="center", va="center", fontsize=10)
ax0.annotate("", xy=(0.23, 0.66), xytext=(0.23, 0.56), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.23, 0.40), xytext=(0.23, 0.30), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.56, 0.56), xytext=(0.40, 0.30), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.set_title("Model Diagram")

ax[1].plot(x_space, E_x, label="E(x, t=t0)")
ax[1].plot(x_space, H_x, label="H(x, t=t0)")
ax[1].set_title("Spatial Trivial-Mode Snapshot")
ax[1].set_xlabel("position x")
ax[1].set_ylabel("field amplitude")
ax[1].legend()

ax[2].plot(t_time, E_t_mid, label="E(L/2, t)")
ax[2].plot(t_time, H_t_mid, label="H(L/2, t)")
ax[2].set_title("Temporal Oscillation at Midpoint")
ax[2].set_xlabel("time t")
ax[2].set_ylabel("field amplitude")
ax[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
def try_read_public_csv(url: str):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        if 'text/csv' in r.headers.get('Content-Type', '') or ',' in r.text[:2000]:
            return pd.read_csv(io.StringIO(r.text))
    except Exception:
        return None
    return None

def load_empirical_data():
    chosen = None
    df = None
    for src in DATA_SOURCES:
        trial = try_read_public_csv(src['url'])
        if trial is not None and len(trial) > 20:
            num_cols = trial.select_dtypes(include=[np.number]).columns.tolist()
            if len(num_cols) >= 1:
                df = trial.copy()
                chosen = src
                break

    if df is None:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (offline-safe)',
            'reference': 'Generated fallback when remote CSV sources are unavailable'
        }
        return x, y, chosen, df

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 1:
        x = np.arange(len(df), dtype=float)
        y = df[numeric_cols[0]].to_numpy(dtype=float)
    else:
        x = df[numeric_cols[0]].to_numpy(dtype=float)
        y = df[numeric_cols[1]].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size < 8 or y.size < 8:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (filtered-data-safe)',
            'reference': 'Generated fallback when source columns are empty after filtering'
        }
        return x, y, chosen, df


    if len(x) > 2000:
        step = max(1, len(x) // 2000)
        x = x[::step]
        y = y[::step]

    if np.ptp(x) == 0:
        x = np.arange(len(x), dtype=float)

    x = (x - np.min(x)) / max(np.ptp(x), 1e-12)
    return x, y, chosen, df

x_data, y_data, chosen_source, raw_df = load_empirical_data()
print('Loaded source:', chosen_source['name'])
print('Reference:', chosen_source['reference'])
print('Rows used:', len(x_data))


In [ ]:
def model_function(x, p1, p2, p3, p4):
    if MODEL_FAMILY == 'oscillator':
        A, gamma, omega, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) * np.cos(omega * x) + c
    if MODEL_FAMILY == 'damped':
        A, gamma, _, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) + c
    if MODEL_FAMILY == 'logistic':
        K, r, x0, c = p1, p2, p3, p4
        return K / (1.0 + np.exp(-r * (x - x0))) + c
    if MODEL_FAMILY == 'michaelis':
        vmax, km, _, c = p1, p2, p3, p4
        return vmax * x / (km + x + 1e-12) + c
    if MODEL_FAMILY == 'diffusion':
        A, D, _, c = p1, p2, p3, p4
        return A * np.exp(-D * x) + c
    return p1 * x + p2

p0 = [np.nanmax(y_data) - np.nanmin(y_data), 0.5, 6.0, np.nanmean(y_data)]
bounds = ([-1e6, 0.0, -1e3, -1e6], [1e6, 50.0, 1e3, 1e6])

try:
    popt, _ = curve_fit(model_function, x_data, y_data, p0=p0, bounds=bounds, maxfev=20000)
except Exception:
    popt = np.array(p0, dtype=float)

y_hat = model_function(x_data, *popt)
rmse = float(np.sqrt(np.mean((y_data - y_hat) ** 2)))
r2 = float(1.0 - np.sum((y_data - y_hat) ** 2) / max(np.sum((y_data - np.mean(y_data)) ** 2), 1e-12))

print('Fitted parameters:', popt)
print('RMSE:', rmse)
print('R^2:', r2)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_data, y_data, s=12, alpha=0.5, label='Empirical data')
ax.plot(x_data, y_hat, color='crimson', linewidth=2.0, label='Fitted surrogate model')
ax.set_title(f'{DOMAIN_TITLE}: empirical fit ({MODEL_FAMILY})')
ax.set_xlabel('Normalized independent variable')
ax.set_ylabel('Observed variable')
ax.legend()
plt.show()


## Interpreting Model Evaluation Trends and Values

Use this guide when reading fitted parameters, RMSE, and $R^2$ outputs:

1. **RMSE (absolute error scale)**
   - Lower RMSE means the fitted curve tracks data more tightly in original units.
   - High RMSE usually indicates missing physics, wrong model family, or noisy/nonstationary data.

2. **$R^2$ (explained variance)**
   - Near 1.0: surrogate captures dominant trend/oscillation structure.
   - Near 0.0: surrogate explains little beyond the mean level.
   - Negative: model is worse than using a constant mean prediction.

3. **Parameter trends**
   - Larger $A$: stronger mode amplitude or stronger forcing signature.
   - Larger $\gamma$: faster decay / stronger damping / shorter memory.
   - Larger $\omega$: faster oscillation or higher characteristic frequency.
   - Offset $c_0$: baseline bias or slowly varying background level.

4. **Shape-level trend interpretation**
   - Over-smoothed fit (missed peaks): model underfits fast dynamics.
   - Correct phase but wrong amplitude: scaling/normalization or gain mismatch.
   - Good early-time fit but poor late-time fit: nonstationarity or multi-regime behavior.

5. **Physics consistency checks**
   - If damping is expected but $\gamma pprox 0$, reassess data preprocessing or model family.
   - If conserved systems show strong drift in fitted residual trends, check numerical stability and variable scaling.
   - Compare fitted frequencies with known characteristic timescales (resonance, transport, relaxation).

## Interactive Simulation Controls

Use the sliders below to perturb model parameters and observe how well the
simulation matches the empirical measurements.


In [ ]:
def interactive_plot(p1, p2, p3, p4):
    y_sim = model_function(x_data, p1, p2, p3, p4)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(x_data, y_data, s=10, alpha=0.45, label='Empirical data')
    ax.plot(x_data, y_sim, color='navy', linewidth=2.0, label='Simulation')
    ax.set_title(f'{DOMAIN_TITLE}: interactive parameter sweep')
    ax.set_xlabel('Normalized independent variable')
    ax.set_ylabel('Observed variable')
    ax.legend()
    plt.show()

interact(
    interactive_plot,
    p1=widgets.FloatSlider(value=float(popt[0]), min=-2*abs(float(popt[0]))-1, max=2*abs(float(popt[0]))+1, step=0.01),
    p2=widgets.FloatSlider(value=float(popt[1]), min=0.0, max=max(1.0, 3*abs(float(popt[1]))+1), step=0.01),
    p3=widgets.FloatSlider(value=float(popt[2]), min=-20.0, max=20.0, step=0.05),
    p4=widgets.FloatSlider(value=float(popt[3]), min=float(np.min(y_data))-2, max=float(np.max(y_data))+2, step=0.01),
);
